## ▶️ Step 0 — set up the notebook (run this first!)

This notebook runs in **Google Colab**. Just press ▶ on the cell below and wait
for the green **✅ Setup complete**, then run the rest top to bottom.

When it asks to **connect Google Drive**, click **Connect** — that lets the data
file download **only once** (it's saved to your Drive and reused by every
notebook) and saves your figures for your poster. You *can* skip it, but then each
notebook re-downloads the ~470 MB data and your figures won't be saved.


In [ ]:
#@title ▶️ Run me first — set up the notebook  { display-mode: "form" }
# Press the ▶ button. (Double-click the title to see the code.)
import os

print("1/3  installing libraries ...")
get_ipython().system('pip install -q "mne==1.10.1" gdown')

print("2/3  downloading the camp toolbox ...")
get_ipython().system('wget -q -O camp_utils.py https://raw.githubusercontent.com/anarghya-das/decoding-the-brain-camp/main/camp_utils.py')

# Connect Drive so the data is downloaded ONCE (saved to your Drive) and your
# figures persist. If you skip it, we fall back to temporary storage.
print("3/3  connecting Google Drive ...")
try:
    from google.colab import drive
    drive.mount("/content/drive")
    data_dir = "/content/drive/MyDrive/DecodingBrain_data"
    os.environ["CAMP_OUTPUT_DIR"] = "/content/drive/MyDrive/DecodingBrain_outputs"
    saved = True
except Exception:
    data_dir = "data"                     # temporary (re-downloads each session)
    os.environ["CAMP_OUTPUT_DIR"] = "outputs"
    saved = False

os.makedirs(data_dir, exist_ok=True)
data_path = os.path.join(data_dir, "synapse_preprocessed.pkl")
os.environ["CAMP_DATA_PATH"] = data_path

if os.path.exists(data_path):
    print("     data already saved in your Drive — skipping download \u26a1")
else:
    print("     downloading the data (~470 MB, one time only) ...")
    import gdown
    gdown.download(id="1Z-NENlKMjL-kL-N46lQ8QA1AbGM7bJHY", output=data_path, quiet=False)

print("\n\u2705 Setup complete.",
      "Data + figures are saved in your Drive (DecodingBrain_*)." if saved
      else "Heads up: you skipped Drive, so the data re-downloads each session.")


# Week 1 · Day 5 — First Group Comparison  🏁 **Checkpoint 1**

Time to put Week 1 together. You'll sweep **every task** and **every band**,
compute the EXP-vs-CTRL difference, and make one big picture: a **heatmap** that
shows, at a glance, where the brains differ most.

This is a mini version of a real figure from the SYNAPSE paper.

### Checkpoint 1 goal
Produce a clean heatmap (tasks × bands) of the EXP−CTRL difference, save it, and
be able to explain what the brightest cell means.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import camp_utils as cu

data = cu.load_camp_data(verbose=False)

## 1. A reusable "compute one feature for a whole group" helper
We'll need this constantly: given a group, task, and band, return the list of
each subject's dB change. Read it, then run it.

In [ ]:
def group_band_powers(data, group, task, band, period="full_stim"):
    """List of baseline-normalized band powers (dB), one per subject in the group."""
    window = cu.get_time_windows(task)[period]
    values = []
    for subject, ep in cu.iter_subjects(data, group, task):
        db = cu.band_power_db(ep, band, window)
        if not np.isnan(db):
            values.append(db)
    return values

# quick test
print("EXP LET alpha:", np.round(group_band_powers(data, "exp", "let", "alpha"), 2))

## 2. Build the difference table
For each task and band we compute the **mean EXP value minus the mean CTRL
value**. A positive number means the EXP (sound-sensitive) group has *more* of
that rhythm than controls during the sound.

### ✏️ Your turn #1 — fill in the inner calculation
The double loop is set up. You compute the two group means and their difference.

In [ ]:
diff_table = pd.DataFrame(index=cu.BAND_ORDER, columns=[t.upper() for t in cu.TASKS],
                          dtype=float)

for task in cu.TASKS:
    for band in cu.BAND_ORDER:
        exp_vals = group_band_powers(data, "exp", task, band)
        ctrl_vals = group_band_powers(data, "ctrl", task, band)

        # TODO: compute the difference of the means.
        # hint: np.mean(exp_vals) - np.mean(ctrl_vals)
        difference = np.nan   # replace this

        diff_table.loc[band, task.upper()] = difference

print(diff_table.round(2))

# check the table got filled with real numbers
cu.check(diff_table.notna().all().all() and diff_table.abs().sum().sum() > 0,
         "Difference table is filled in.",
         "Every cell should be np.mean(exp_vals) - np.mean(ctrl_vals).")

## 3. The heatmap
A heatmap colors each cell by its value. We use a **diverging** colormap
(`RdBu_r`): red = EXP higher, blue = CTRL higher, white = no difference. We
center the color scale at 0 so the colors are meaningful.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

vmax = np.nanmax(np.abs(diff_table.values))   # symmetric limits around 0
im = ax.imshow(diff_table.values.astype(float), cmap="RdBu_r",
               vmin=-vmax, vmax=vmax, aspect="auto")

ax.set_xticks(range(len(cu.TASKS)))
ax.set_xticklabels([t.upper() for t in cu.TASKS])
ax.set_yticks(range(len(cu.BAND_ORDER)))
ax.set_yticklabels(cu.BAND_ORDER)
ax.set_title("EXP − CTRL band-power difference (dB)")

# write the number in each cell
for i in range(len(cu.BAND_ORDER)):
    for j in range(len(cu.TASKS)):
        ax.text(j, i, f"{diff_table.iloc[i, j]:+.2f}",
                ha="center", va="center", fontsize=9)

cbar = fig.colorbar(im, ax=ax)
cbar.set_label("EXP higher  ←→  CTRL higher")
plt.tight_layout()
plt.savefig(cu.save_path("checkpoint1_heatmap.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved to outputs/checkpoint1_heatmap.png")

### ✏️ Your turn #2 — read your own figure
Find the cell with the **largest absolute difference** (most red or most blue).
We can find it with code:

In [ ]:
flat = diff_table.abs().stack()
band_max, task_max = flat.idxmax()
value = diff_table.loc[band_max, task_max]
print(f"Biggest group difference: {band_max} during {task_max} = {value:+.2f} dB")

# TODO: write 1–2 sentences (as a comment) answering:
#   - Which group has MORE of this rhythm (red=EXP, blue=CTRL)?
#   - Why might THIS task and THIS band show the biggest difference?
# YOUR ANSWER:
#

## 4. Reality check (important!)
A difference in the means looks exciting, but with only 18 vs 10 people it could
easily be noise. Look at the spread of individual subjects behind one cell:

In [ ]:
exp_vals = group_band_powers(data, "exp", task_max.lower(), band_max)
ctrl_vals = group_band_powers(data, "ctrl", task_max.lower(), band_max)

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(np.random.uniform(0.9, 1.1, len(exp_vals)), exp_vals,
           color=cu.EXP_COLOR, label="EXP", alpha=0.8)
ax.scatter(np.random.uniform(1.9, 2.1, len(ctrl_vals)), ctrl_vals,
           color=cu.CTRL_COLOR, label="CTRL", alpha=0.8)
ax.set_xticks([1, 2]); ax.set_xticklabels(["EXP", "CTRL"])
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax.set_ylabel(f"{band_max} change (dB) — {task_max}")
ax.set_title("Do the dots actually separate, or do they overlap a lot?")
ax.legend()
plt.show()

**Discuss:** Do the orange and blue dots clearly separate, or do they overlap?
This is exactly *why* Week 2 exists: we need **statistics** to decide whether a
difference is real, and **effect sizes** to decide whether it's big enough to
matter.

## 🏁 Checkpoint 1 — show your instructor
- [ ] Your heatmap is saved in `outputs/checkpoint1_heatmap.png`
- [ ] You can point to the biggest-difference cell and say which group is higher
- [ ] You can explain why we shouldn't trust a difference of means by itself

➡️ **Next week:** we turn these pictures into rigorous numbers — features,
statistical tests, effect sizes, and publication-quality figures.